<a href="https://colab.research.google.com/github/gokulbytes/personalized-news-recommendation-engine/blob/main/main_(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Requirements

In [1]:
# Install necessary libraries
! pip install pandas numpy scikit-learn -q

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# main

In [3]:
# Load the dataset into a pandas DataFrame
df=pd.read_csv('/content/final_df.csv')
# Display the first 5 rows of the DataFrame
df.head()

,UserID,NewsID,URL,Content,Clicked
0,U13740,N55689,https://assets.msn.com/labs/mind/BBWAPO6.html,"Charles Rogers, former Michigan State football...",1
1,U91836,N17059,https://assets.msn.com/labs/mind/BBWE1bu.html,No. 1 milk company declares bankruptcy amid dr...,1
2,U73700,N23814,https://assets.msn.com/labs/mind/AAHr37p.html,I moved from the US to the UK. Here are the 8 ...,1
3,U34670,N49685,https://assets.msn.com/labs/mind/BBWyk8E.html,Broadway Star Laurel Griggs Suffered Asthma At...,1
4,U8125,N8400,https://assets.msn.com/labs/mind/BBOA5nc.html,These are the new cars that depreciate least D...,1


In [4]:
# Drop the 'Content' column as it is not needed for the recommendation system
df = df.drop(columns=['Content'])

## Data Transformation

In [5]:
# Create a user-item matrix using pivot_table
# Rows represent users, columns represent news articles, and values indicate if a user clicked on an article (1) or not (0)
user_item_matrix = df.pivot_table(index='UserID', columns='NewsID', values='Clicked', fill_value=0)
# Display the first few rows of the user-item matrix
user_item_matrix.head()

NewsID,N10032,N10051,N10056,N10057,N1006,N10061,N10067,N10070,N10072,N10073,...,N9923,N9929,N9944,N9950,N9961,N9989,N999,N9991,N9997,N9999
UserID,,,,,,,,,,,,,,,,,,,,,
U100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
U1000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
U10001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
U10003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
U10008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
# Display the shape of the user-item matrix (number of users, number of news articles)
user_item_matrix.shape

(50000, 7713)

### reduced_user_item_matrix

In [7]:
# Reduce the user-item matrix to the first 10000 users for computational efficiency
reduced_user_item_matrix = user_item_matrix.iloc[:10000]

## Cosine Similarity

In [8]:
# Transpose the reduced user-item matrix to create an item-user matrix
item_user_matrix = reduced_user_item_matrix.T

In [9]:
# Calculate the cosine similarity between items
item_similarity = cosine_similarity(item_user_matrix)
# Convert the similarity matrix to a pandas DataFrame with item IDs as index and columns
item_similarity_df = pd.DataFrame(item_similarity, index=item_user_matrix.index, columns=item_user_matrix.index)

## Recommendation Function

In [10]:
def recommend_items_item_based(user_id, user_item_matrix, item_similarity_df, top_n=5):
    # Check if the user exists in the matrix
    if user_id not in user_item_matrix.index:
        return f"User '{user_id}' not found in matrix."

    # Get the items clicked by the user
    clicked_items = user_item_matrix.loc[user_id]
    clicked_items = clicked_items[clicked_items > 0].index.tolist()

    # If the user hasn't clicked any items, return a message
    if not clicked_items:
        return f"User '{user_id}' has not clicked any items."

    # Initialize a series to store recommendation scores
    scores = pd.Series(dtype=np.float64, index=item_similarity_df.columns)

    # Calculate scores for each item based on similarity to clicked items
    for item in clicked_items:
        similar_items = item_similarity_df[item]
        scores = scores.add(similar_items, fill_value=0)

    # Remove items that the user has already clicked from the scores
    scores = scores.drop(labels=clicked_items, errors='ignore')

    # Return the top N recommended items based on scores
    return scores.sort_values(ascending=False).head(top_n)

### Testing

In [11]:
# Define the test user ID from the first user in the reduced user-item matrix
test_user = reduced_user_item_matrix.index[0]

# Get item-based recommendations for the test user
item_based_recs = recommend_items_item_based(user_id=test_user, user_item_matrix=reduced_user_item_matrix, item_similarity_df=item_similarity_df, top_n=5)

# Print the top 5 item-based recommendations for the test user
print(f"Item-Based Top 5 Recommendations for User {test_user}:\n")
print(item_based_recs)

Item-Based Top 5 Recommendations for User U100:

NewsID
N36273    0.288675
N60858    0.166667
N22339    0.078567
N60105    0.077152
N43102    0.075810
dtype: float64


## Save Model

In [12]:
import joblib

joblib.dump(user_item_matrix, 'collaborative_filtering.pkl')

print("Model saved successfully")

Model saved successfully
